# Vision Language Models for Semantic-Aware Radiology Report Generation

## Motivation

Radiology report generation is an important application of vision-language models in healthcare. The goal is to automatically generate clinically meaningful reports from chest X-ray images. This can help reduce reporting time, support radiologists, and improve accessibility in medical systems with limited resources.

Most existing approaches focus heavily on improving traditional evaluation metrics such as BLEU and ROUGE. However, in medical text generation, exact wording is not always the best indicator of correctness. Two reports can describe the same clinical condition using different terminology while receiving a low lexical score.

For example:

- "cardiomegaly observed"
- "enlarged heart detected"

These statements are semantically similar, but lexical metrics may still penalize them because the wording differs.

Another major issue in radiology report generation is hallucination. Models may generate findings that are not visible in the image or repeat medically unsupported statements. This creates a serious challenge because fluent language does not always mean clinically correct language.

Because of this, semantic correctness and clinical consistency become more important than exact word overlap alone.

In this project, the focus shifts from only improving lexical metrics toward understanding and evaluating semantic alignment in generated reports. Alongside standard evaluation metrics, we explore:

- semantic-aware evaluation
- clinically-aware scoring
- hallucination analysis
- latent feature interpretation
- attention-based grounding analysis

The objective is not only to generate fluent reports, but also to better understand whether the generated reports preserve meaningful clinical information.

## 1. Environment Setup

In [ ]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms

from torch.utils.data import Dataset, DataLoader
from torchvision import models

warnings.filterwarnings("ignore")

#### Reproducibility Setup

Deep learning experiments can produce slightly different results across runs because of random initialization, shuffling, and GPU operations.

To improve reproducibility, fixed random seeds are used across:
- Python
- NumPy
- PyTorch

This helps make experiments more stable and easier to compare during ablation studies and evaluation.

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Random seed set to: {SEED}")


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("GPU Count:", torch.cuda.device_count())

### Project Pipeline Overview

The overall pipeline used in this project consists of:

1. Chest X-ray preprocessing
2. Report text preprocessing
3. CNN-based image feature extraction
4. Attention-based report generation
5. Semantic-aware evaluation
6. Interpretability and hallucination analysis

The baseline architecture uses:
- ResNet18 as the visual encoder
- Bahdanau attention
- GRU decoder

Additional improvements explored include:
- beam search decoding
- coverage loss
- staged fine-tuning
- semantic-aware evaluation metrics
- clinically-aware scoring methods

## 2. Dataset Preparation



#### Indiana University Chest X-ray Dataset

This project uses the Indiana University Chest X-ray dataset, which is a commonly used benchmark for radiology report generation research.

The dataset contains:
- frontal and lateral chest X-ray images
- paired radiology reports
- findings and impression sections
- study-level identifiers

Each sample consists of:
1. a chest X-ray image
2. a corresponding radiology report

Unlike standard image classification datasets, this task requires the model to learn both:
- visual understanding
- clinically meaningful language generation

The dataset is relatively small compared to large-scale medical datasets such as MIMIC-CXR. Because of this, careful preprocessing and data splitting become especially important to avoid data leakage and overfitting.

In [ ]:
IMAGE_DIR = "/content/images/images_normalized/"
REPORT_PATH = "/content/indiana_reports.csv"

print("Image Directory:", IMAGE_DIR)
print("Report File:", REPORT_PATH)

### 2.1 Dataset Overview

Before training the model, it is important to understand:
- dataset size
- report structure
- vocabulary distribution
- image-report pairing

This helps identify:
- missing values
- duplicated studies
- imbalance issues
- preprocessing requirements

In [ ]:
df = pd.read_csv(REPORT_PATH)

print("Dataset Shape:", df.shape)
df.head()

In [ ]:
print("Number of Samples:", len(df))

print("\nColumns:")
print(df.columns)

print("\nMissing Values:")
print(df.isnull().sum())

### Paired Image-Report Structure

Each row in the dataset contains:
- an image filename
- a corresponding radiology report

The model learns to map:
- visual chest X-ray features
to
- clinically meaningful textual descriptions

This paired structure forms the foundation of the vision-language learning pipeline.

sample = df.iloc[0]

print("Image Path:")
print(sample['image_path'])

print("\nReport:")
print(sample['report'])

### Report Length Analysis

Radiology reports vary in length depending on:
- disease complexity
- number of findings
- reporting style

Understanding report length distribution helps determine:
- maximum sequence length
- padding strategy
- decoder efficiency

In [ ]:
df['report_length'] = df['report'].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(8,5))
plt.hist(df['report_length'], bins=30)

plt.xlabel("Report Length")
plt.ylabel("Frequency")
plt.title("Distribution of Report Lengths")

plt.show()

### 2.2 UID-Level Splitting

One major issue in medical imaging research is data leakage.

If images from the same study appear in both training and testing sets, the model may memorize patterns rather than genuinely generalize.

To avoid this, splitting is performed at the study or UID level instead of random image-level splitting.

This ensures:
- cleaner evaluation
- better generalization testing
- more reliable performance estimates

In [ ]:
from sklearn.model_selection import train_test_split

unique_uids = df['uid'].unique()

train_uids, temp_uids = train_test_split(
    unique_uids,
    test_size=0.2,
    random_state=SEED
)

val_uids, test_uids = train_test_split(
    temp_uids,
    test_size=0.5,
    random_state=SEED
)

train_df = df[df['uid'].isin(train_uids)]
val_df = df[df['uid'].isin(val_uids)]
test_df = df[df['uid'].isin(test_uids)]

In [ ]:
# split stats

print("Training Samples:", len(train_df))
print("Validation Samples:", len(val_df))
print("Testing Samples:", len(test_df))

print("\nUnique Training UIDs:", train_df['uid'].nunique())
print("Unique Validation UIDs:", val_df['uid'].nunique())
print("Unique Testing UIDs:", test_df['uid'].nunique())

### Why UID-Level Splitting Matters

This step is especially important in radiology datasets because:
- multiple images may belong to the same patient or study
- duplicated patterns can artificially inflate metrics
- leakage creates unrealistic evaluation performance

UID-level splitting helps ensure that the model is evaluated on genuinely unseen studies.

### 2.3 Image Loading and Preprocessing

Medical images require careful preprocessing before being passed into the encoder network.

The preprocessing pipeline includes:
- image resizing
- tensor conversion
- normalization

Images are resized to a fixed resolution to ensure consistent input dimensions for the CNN encoder.

In [ ]:
IMAGE_SIZE = 224

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# normalisation 

from PIL import Image

sample_image_path = os.path.join(
    IMAGE_DIR,
    train_df.iloc[0]['image_path']
)

image = Image.open(sample_image_path).convert("RGB")

plt.figure(figsize=(5,5))
plt.imshow(image)
plt.title("Sample Chest X-ray")
plt.axis("off")

plt.show()

In [ ]:
image_tensor = transform(image)

print("Tensor Shape:", image_tensor.shape)

- Normalization helps stabilize training by scaling pixel intensities into a more consistent range.

- The ImageNet normalization statistics are used because the encoder backbone is initialized using pretrained ImageNet weights.

### Final Preprocessing Pipeline

At this stage:
- reports have been paired with images
- UID-level splitting has been completed
- image preprocessing has been defined
- transformed tensors are ready for model training

The next stage focuses on:
- text preprocessing
- vocabulary construction
- tokenization
- sequence preparation
for the decoder network.

## 3. Exploratory Data Analysis (EDA)

Before training the model, it is important to understand the dataset beyond just loading images and reports into memory.

In medical vision-language tasks, dataset structure can strongly influence:
- language generation quality
- vocabulary learning
- semantic consistency
- class imbalance
- hallucination behavior

This section explores:
- report length distribution
- vocabulary characteristics
- common clinical terminology
- imbalance between normal and abnormal studies
- sample image-report pairs

Understanding these patterns helps guide later decisions involving:
- tokenizer design
- sequence length selection
- training strategy
- evaluation methodology